# Final Demo - One-Year Historical Pipeline Evidence

This notebook is the final proof that the 2024 historical dataset pipeline has already produced usable one-year lakehouse outputs. PR1/PR2 prove infrastructure and monthly pipeline pieces; this notebook validates the one-year state from MinIO and Spark.

Safety rule: this notebook is read-only by default. It inventories raw/Bronze/Silver/Gold outputs, runs read-only Spark counts on Silver and Gold, prints the optional one-year rebuild command, and calls the prediction API with one historical Gold row. It does **not** rewrite MinIO or retrain models unless `RUN_FULL_YEAR_PIPELINE=True` is changed intentionally.


In [1]:
YEAR = 2024
RUN_FULL_YEAR_PIPELINE = False
WITH_DELTA = False
WITH_FINAL_MODEL = False
RUN_SPARK_ROW_COUNTS = True
CALL_HISTORICAL_GOLD_ROW_API = True
API_URL = 'http://aviation-api:3000/predict'

print('Year:', YEAR)
print('Read-only validation mode:', not RUN_FULL_YEAR_PIPELINE)
print('Run Spark Silver/Gold row counts:', RUN_SPARK_ROW_COUNTS)
print('Run historical Gold-row API call:', CALL_HISTORICAL_GOLD_ROW_API)


Year: 2024
Read-only validation mode: True
Run Spark Silver/Gold row counts: True
Run historical Gold-row API call: True


## 1. One-Year Raw And Lakehouse Inventory

This cell verifies month-by-month objects and storage size in MinIO for the 2024 pipeline. It checks raw ARCO-ERA5 weather, raw BTS ZIPs, Bronze weather, Bronze BTS, Silver flight-weather daily rows, Gold training features, and Gold Delta transaction-log evidence.


In [2]:
import os
import boto3
import pandas as pd
from IPython.display import display

endpoint = os.environ['MINIO_ENDPOINT_INTERNAL']
s3 = boto3.client(
    's3',
    endpoint_url=endpoint,
    aws_access_key_id=os.environ['MINIO_ROOT_USER'],
    aws_secret_access_key=os.environ['MINIO_ROOT_PASSWORD'],
    region_name=os.environ.get('AWS_REGION', 'us-east-1'),
)

def object_summary(bucket: str, prefix: str):
    count = 0
    size = 0
    paginator = s3.get_paginator('list_objects_v2')
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for item in page.get('Contents', []):
            count += 1
            size += int(item.get('Size', 0))
    return count, size

def gib(value: int) -> float:
    return round(value / (1024 ** 3), 3)

rows = []
for month in range(1, 13):
    mm = f'{month:02d}'
    raw_weather_count, raw_weather_bytes = object_summary('raw', f'arco_era5_us_airport_hourly/year={YEAR}/month={mm}/')
    raw_bts_count, raw_bts_bytes = object_summary('raw', f'bts_on_time/raw_zip/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_{YEAR}_{month}.zip')
    bronze_weather_count, bronze_weather_bytes = object_summary('lakehouse', f'bronze/weather/year={YEAR}/month={mm}/')
    bronze_bts_count, bronze_bts_bytes = object_summary('lakehouse', f'bronze/bts_on_time/year={YEAR}/month={mm}/')
    silver_count, silver_bytes = object_summary('lakehouse', f'silver/flight_weather_daily/year={YEAR}/month={mm}/')
    gold_count, gold_bytes = object_summary('lakehouse', f'gold/training_features/year={YEAR}/month={mm}/')
    gold_delta_log_count, gold_delta_log_bytes = object_summary('lakehouse', f'gold_delta/training_features/year={YEAR}/month={mm}/_delta_log/')
    rows.append({
        'month': mm,
        'raw_weather_objects': raw_weather_count,
        'raw_weather_gib': gib(raw_weather_bytes),
        'raw_bts_zip_objects': raw_bts_count,
        'bronze_weather_objects': bronze_weather_count,
        'bronze_weather_gib': gib(bronze_weather_bytes),
        'bronze_bts_objects': bronze_bts_count,
        'silver_objects': silver_count,
        'gold_objects': gold_count,
        'gold_delta_log_objects': gold_delta_log_count,
    })

inventory_df = pd.DataFrame(rows)
display(inventory_df)
print('Total raw weather GiB for year:', round(inventory_df['raw_weather_gib'].sum(), 2))
print('Total Bronze weather GiB for year:', round(inventory_df['bronze_weather_gib'].sum(), 2))


,month,raw_weather_objects,raw_weather_gib,raw_bts_zip_objects,bronze_weather_objects,bronze_weather_gib,bronze_bts_objects,silver_objects,gold_objects,gold_delta_log_objects
0,01,7,0.376,1,6,0.739,3,4,3,3
1,02,7,0.345,1,6,0.697,3,4,3,3
2,03,7,0.380,1,6,0.750,3,4,3,3
3,04,7,0.368,1,6,0.724,3,4,3,3
4,05,7,0.387,1,6,0.754,4,4,3,3
5,06,7,0.370,1,6,0.723,4,4,3,3
6,07,7,0.380,1,6,0.741,4,4,3,3
7,08,7,0.378,1,6,0.733,4,4,3,3
8,09,7,0.357,1,6,0.708,3,4,3,3
9,10,7,0.356,1,6,0.719,4,4,3,3


Total raw weather GiB for year: 4.44
Total Bronze weather GiB for year: 8.76


## 2. One-Year Completeness Gates

These assertions make the notebook useful as proof, not just a table. If a month is missing from any required layer, this cell fails immediately.


In [3]:
required_positive_columns = [
    'raw_weather_objects',
    'raw_bts_zip_objects',
    'bronze_weather_objects',
    'bronze_bts_objects',
    'silver_objects',
    'gold_objects',
    'gold_delta_log_objects',
]

missing = inventory_df.loc[(inventory_df[required_positive_columns] <= 0).any(axis=1), ['month'] + required_positive_columns]
if not missing.empty:
    display(missing)
    raise AssertionError('One-year pipeline completeness check failed for one or more months.')

print('PASS: all 12 months have raw, Bronze, Silver, Gold, and Gold Delta-log evidence.')
print('Months validated:', ', '.join(inventory_df['month']))


PASS: all 12 months have raw, Bronze, Silver, Gold, and Gold Delta-log evidence.
Months validated: 01, 02, 03, 04, 05, 06, 07, 08, 09, 10, 11, 12


## 3. Spark Row Counts For One-Year Silver And Gold

This is read-only Spark validation over the generated 2024 lakehouse outputs. It counts Silver and Gold rows by month, proving the final-year tables are queryable by Spark. Bronze weather row counts are intentionally not recomputed here because that table is much larger and the object inventory above already proves the monthly Bronze partitions exist.


In [4]:
if RUN_SPARK_ROW_COUNTS:
    import sys
    from pathlib import Path
    from pyspark.sql import functions as F

    PROJECT_ROOT = Path('/workspace')
    if str(PROJECT_ROOT) not in sys.path:
        sys.path.insert(0, str(PROJECT_ROOT))

    from spark_jobs.common import create_spark_session, load_settings, s3a_uri

    settings = load_settings()
    spark = create_spark_session('final-one-year-silver-gold-validation', settings)
    spark.sparkContext.setLogLevel('WARN')

    def month_column(df):
        if 'month' in df.columns:
            return F.lpad(F.col('month').cast('string'), 2, '0')
        return F.regexp_extract(F.input_file_name(), r'month=(\d+)', 1)

    silver_path = s3a_uri(settings.lakehouse_bucket, f'silver/flight_weather_daily/year={YEAR}')
    gold_path = s3a_uri(settings.lakehouse_bucket, f'gold/training_features/year={YEAR}')

    silver_df = spark.read.parquet(silver_path)
    gold_df = spark.read.parquet(gold_path)

    silver_monthly = (
        silver_df.withColumn('month_key', month_column(silver_df))
        .groupBy('month_key')
        .agg(F.count('*').alias('silver_rows'))
        .orderBy('month_key')
    )
    gold_monthly = (
        gold_df.withColumn('month_key', month_column(gold_df))
        .groupBy('month_key')
        .agg(
            F.count('*').alias('gold_rows'),
            F.sum(F.col('label').cast('double')).alias('disrupted_rows'),
            F.avg(F.col('label').cast('double')).alias('disruption_rate'),
        )
        .orderBy('month_key')
    )

    print('Silver rows by month:')
    silver_monthly.show(12, truncate=False)
    print('Gold rows and labels by month:')
    gold_monthly.show(12, truncate=False)
    print('Total Silver rows:', silver_df.count())
    print('Total Gold rows:', gold_df.count())

    spark.stop()
else:
    print('Skipped Spark row counts. Set RUN_SPARK_ROW_COUNTS=True to validate one-year Silver/Gold tables.')


Silver rows by month:


+---------+-----------+
|month_key|silver_rows|
+---------+-----------+
|01       |543121     |
|02       |515584     |
|03       |587509     |
|04       |578167     |
|05       |605545     |
|06       |606721     |
|07       |630066     |
|08       |615151     |
|09       |579980     |
|10       |612560     |
|11       |571833     |
|12       |586045     |
+---------+-----------+

Gold rows and labels by month:


+---------+---------+--------------+-------------------+
|month_key|gold_rows|disrupted_rows|disruption_rate    |
+---------+---------+--------------+-------------------+
|01       |543121   |145656.0      |0.2681833329957781 |
|02       |515584   |84069.0       |0.16305587450347567|
|03       |587509   |125791.0      |0.2141090604569462 |
|04       |578167   |113126.0      |0.19566319073900793|
|05       |605545   |165426.0      |0.27318531240452815|
|06       |606721   |156456.0      |0.25787141041763845|
|07       |630066   |198866.0      |0.31562725174822953|
|08       |615151   |153253.0      |0.24913070124245917|
|09       |579980   |92765.0       |0.1599451705231215 |
|10       |612560   |86119.0       |0.14058867702755648|
|11       |571833   |85922.0       |0.15025715549819615|
|12       |586045   |126766.0      |0.2163076214283886 |
+---------+---------+--------------+-------------------+



Total Silver rows: 7032282


Total Gold rows: 7032282


## 4. Model Experiment Evidence

The project did test multiple Spark MLlib baselines for January 2024 diagnostics. The final served model is intentionally a practical baseline; the strongest part of this project is the reproducible data pipeline, registry, API, replay, and monitoring path.


In [5]:
import json
from pathlib import Path

metrics_path = Path('/workspace/data/local_cache/model_metrics/january_2024_mllib.json')
if metrics_path.exists():
    metrics = json.loads(metrics_path.read_text())
    metrics_df = pd.DataFrame(metrics).sort_values('auc', ascending=False)
    display(metrics_df[[
        'run_name', 'train_rows', 'test_rows', 'auc', 'accuracy', 'positive_precision', 'positive_recall',
        'true_positive', 'false_positive', 'true_negative', 'false_negative'
    ]])
    best = metrics_df.iloc[0]
    print('Best diagnostic AUC run:', best['run_name'], 'AUC=', round(float(best['auc']), 4))
else:
    print('Metrics file not found:', metrics_path)


,run_name,train_rows,test_rows,auc,accuracy,positive_precision,positive_recall,true_positive,false_positive,true_negative,false_negative
4,random_forest_trees_40_depth_8,434366,108755,0.700825,0.732766,0.772727,0.000585,17,5,79675,29058
0,logistic_regression_reg_0_01,434366,108755,0.690043,0.750228,0.647796,0.144041,4188,2277,77403,24887
1,logistic_regression_reg_0_05,434366,108755,0.684267,0.743423,0.660455,0.082889,2410,1239,78441,26665
2,logistic_regression_reg_0_10,434366,108755,0.679896,0.738136,0.651577,0.044058,1281,685,78995,27794
3,random_forest_trees_20_depth_6,434366,108755,0.677954,0.732656,0.000000,0.000000,0,0,79680,29075


Best diagnostic AUC run: random_forest_trees_40_depth_8 AUC= 0.7008


## 5. Optional Full-Year Runner

This cell shows the exact command for the full one-year pipeline. It reuses the same production Spark modules as PR2 for every month and can optionally build Delta and register a full-year model. Keep this in read-only mode during presentation.


In [6]:
import subprocess, sys

command = [
    sys.executable, '-m', 'spark_jobs.run_year_pipeline',
    '--year', str(YEAR),
]
if WITH_DELTA:
    command.append('--with-delta')
if WITH_FINAL_MODEL:
    command.append('--with-final-model')

print('Full-year command:')
print(' '.join(command))
if RUN_FULL_YEAR_PIPELINE:
    subprocess.run(command, check=True, cwd='/workspace')
else:
    print('Safe read-only mode: command shown but not executed during presentation.')


Full-year command:
/opt/conda/bin/python -m spark_jobs.run_year_pipeline --year 2024
Safe read-only mode: command shown but not executed during presentation.


## 6. API Call From a Historical Gold Feature Row

This cell calls the deployed API with one historical Gold feature row from the downloaded lakehouse and captures the response inside the notebook output. It is safe to rerun: it reads one row and sends one API request.


In [7]:
import subprocess, sys

if CALL_HISTORICAL_GOLD_ROW_API:
    completed = subprocess.run([
        sys.executable, '-m', 'spark_jobs.call_api_with_gold_sample',
        '--year', str(YEAR),
        '--month', '1',
        '--api-url', API_URL,
    ], text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, cwd='/workspace')
    print(completed.stdout)
    if completed.returncode != 0:
        raise RuntimeError(f'Gold-row API call failed with exit code {completed.returncode}')
else:
    print('Skipped historical Gold-row API call.')


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/22 03:00:29 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/22 03:00:30 WARN SparkConf: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in standalone/kubernetes and LOCAL_DIRS in YARN).
26/06/22 03:00:51 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties

[Stage 0:>                                                          (0 + 0) / 1]

[Stage 0:>                                                          (0 + 1) / 1]

                                                                                

[Stage 1:>                                                       